In [1]:
import json
from collections import Counter
import os

os.chdir("/home/a67606/Thesis/EXIST_MEMES")

PATH = "./Dataset/memes/training/EXIST2026_training.json"  # ajusta ao teu caminho

with open(PATH, encoding="utf-8") as f:
    data = json.load(f)

print(f"tipo da raiz: {type(data).__name__}")

if isinstance(data, dict):
    items = list(data.values())
    print(f"n itens: {len(items)}")
    print(f"chaves de topo (3 primeiras): {list(data.keys())[:3]}")
else:
    items = data
    print(f"n itens: {len(items)}")

key_freq = Counter()
for it in items:
    if isinstance(it, dict):
        key_freq.update(it.keys())

print(f"\n--- chaves ao nivel do item ({len(key_freq)} distintas) ---")
for k, n in sorted(key_freq.items()):
    exemplo = next((it[k] for it in items if isinstance(it, dict) and k in it), None)
    amostra = repr(exemplo)
    if len(amostra) > 90:
        amostra = amostra[:90] + "..."
    print(f"{k:34s} n={n:6d}  {type(exemplo).__name__:5s}  {amostra}")

tipo da raiz: dict
n itens: 3984
chaves de topo (3 primeiras): ['110887', '110466', '111269']

--- chaves ao nivel do item (17 distintas) ---
age_annotators                     n=  3984  list   ['18-22', '23-45', '46+', '46+', '18-22', '23-45']
annotators                         n=  3984  list   ['Annotator_193', 'Annotator_194', 'Annotator_195', 'Annotator_196', 'Annotator_197', 'Ann...
countries_annotators               n=  3984  list   ['Sweden', 'United States', 'Spain', 'Portugal', 'Spain', 'Mexico']
ethnicities_annotators             n=  3984  list   ['White or Caucasian', 'Multiracial', 'White or Caucasian', 'White or Caucasian', 'Hispano...
gender_annotators                  n=  3984  list   ['F', 'F', 'F', 'M', 'M', 'M']
id_EXIST                           n=  3984  str    '110887'
labels_task2_1                     n=  3984  list   ['YES', 'YES', 'YES', 'YES', 'YES', 'YES']
labels_task2_2                     n=  3984  list   ['JUDGEMENTAL', 'DIRECT', 'DIRECT', 'DIRECT', 'DIREC

In [2]:
sens_users = set()
for it in items:
    s = it.get("sensorial")
    if isinstance(s, dict):
        sens_users.update(s.get("users", []))

ann_users = set()
for it in items:
    ann_users.update(it.get("annotators", []))

print(f"sujeitos sensoriais: {len(sens_users)}  exemplos: {sorted(sens_users)[:6]}")
print(f"anotadores:          {len(ann_users)}")
print(f"interseção:          {len(sens_users & ann_users)}")

sujeitos sensoriais: 13  exemplos: ['EN1', 'EN2', 'EN3', 'EN4', 'EN5', 'EN6']
anotadores:          887
interseção:          0


In [3]:
import numpy as np

def gerar_2pl(n_itens, n_anotadores, votos_por_item, seed):
    """
    Gera respostas binárias sob o modelo IRT 2PL.
    Devolve (respostas, theta, b, a).
    respostas: matriz (n_anotadores x n_itens), NaN onde não houve voto.
    """
    rng = np.random.default_rng(seed)

    theta = rng.normal(0.0, 1.0, size=n_anotadores)
    b     = rng.normal(0.0, 1.0, size=n_itens)
    a     = rng.lognormal(mean=0.0, sigma=0.3, size=n_itens)

    respostas = np.full((n_anotadores, n_itens), np.nan)

    for j in range(n_itens):
        quem = rng.choice(n_anotadores, size=votos_por_item, replace=False)
        logit = a[j] * (theta[quem] - b[j])
        p = 1.0 / (1.0 + np.exp(-logit))
        respostas[quem, j] = (rng.random(votos_por_item) < p).astype(float)

    return respostas, theta, b, a


resp, theta_true, b_true, a_true = gerar_2pl(
    n_itens=1000, n_anotadores=200, votos_por_item=50, seed=42
)

n_por_anotador = np.sum(~np.isnan(resp), axis=1)
n_por_item     = np.sum(~np.isnan(resp), axis=0)

print(f"matriz: {resp.shape}")
print(f"votos totais: {int(np.sum(~np.isnan(resp)))}")
print(f"votos/item:      min={n_por_item.min()}  max={n_por_item.max()}")
print(f"votos/anotador:  min={n_por_anotador.min()}  mediana={np.median(n_por_anotador):.0f}  max={n_por_anotador.max()}")
print(f"densidade da matriz: {np.mean(~np.isnan(resp)):.3f}")
print(f"taxa global de 1s: {np.nanmean(resp):.3f}")
print(f"a: media={a_true.mean():.3f}  min={a_true.min():.3f}  max={a_true.max():.3f}")

matriz: (200, 1000)
votos totais: 50000
votos/item:      min=50  max=50
votos/anotador:  min=213  mediana=250  max=279
densidade da matriz: 0.250
taxa global de 1s: 0.494
a: media=1.021  min=0.401  max=2.397


In [4]:
from scipy.optimize import minimize

def estimar_2pl_mml(respostas, n_pontos=21, max_iter=200, tol=1e-5, verbose=True):
    """
    Estima parâmetros IRT 2PL por MML-EM com quadratura de Gauss-Hermite.
    respostas: (n_anotadores x n_itens), NaN onde não houve voto.
    Devolve (theta_eap, b_hat, a_hat, historico_loglik).
    """
    n_ann, n_it = respostas.shape
    obs = ~np.isnan(respostas)
    R = np.nan_to_num(respostas, nan=0.0)

    nos, pesos = np.polynomial.hermite_e.hermegauss(n_pontos)
    pesos = pesos / pesos.sum()

    a_hat = np.ones(n_it)
    b_hat = np.zeros(n_it)
    historico = []

    for it_num in range(max_iter):
        logit = a_hat[None, :, None] * (nos[None, None, :] - b_hat[None, :, None])
        logp1 = -np.logaddexp(0.0, -logit)
        logp0 = -np.logaddexp(0.0, logit)

        contrib = obs[:, :, None] * (R[:, :, None] * logp1 + (1.0 - R)[:, :, None] * logp0)
        loglik_ann = contrib.sum(axis=1)

        m = loglik_ann.max(axis=1, keepdims=True)
        naolog = np.exp(loglik_ann - m) * pesos[None, :]
        marg = naolog.sum(axis=1, keepdims=True)
        post = naolog / marg
        loglik = float(np.sum(np.log(marg) + m))
        historico.append(loglik)

        n_kj = np.einsum('ik,ij->jk', post, obs.astype(float))
        r_kj = np.einsum('ik,ij->jk', post, R * obs)

        for j in range(n_it):
            def neg(par):
                aj, bj = par
                lg = aj * (nos - bj)
                lp1 = -np.logaddexp(0.0, -lg)
                lp0 = -np.logaddexp(0.0, lg)
                return -np.sum(r_kj[j] * lp1 + (n_kj[j] - r_kj[j]) * lp0)
            res = minimize(neg, x0=[a_hat[j], b_hat[j]], method='L-BFGS-B',
                           bounds=[(0.05, 5.0), (-6.0, 6.0)])
            a_hat[j], b_hat[j] = res.x

        if it_num > 0 and abs(historico[-1] - historico[-2]) < tol:
            if verbose:
                print(f"convergiu em {it_num+1} iteracoes")
            break

    quebras = sum(1 for i in range(1, len(historico)) if historico[i] < historico[i-1] - 1e-6)
    if verbose:
        print(f"loglik final: {historico[-1]:.2f}")
        print(f"quebras de monotonicidade: {quebras} / {len(historico)-1}")

    theta_eap = post @ nos
    return theta_eap, b_hat, a_hat, historico


theta_hat, b_hat, a_hat, hist = estimar_2pl_mml(resp)

convergiu em 132 iteracoes
loglik final: -26872.33
quebras de monotonicidade: 0 / 131


In [5]:
from scipy.stats import pearsonr, spearmanr

def avaliar_recuperacao(theta_true, b_true, a_true, theta_hat, b_hat, a_hat, rotulo=""):
    print(f"--- recuperacao {rotulo} ---")
    linhas = []
    for nome, vt, vh in [("theta", theta_true, theta_hat),
                         ("b    ", b_true, b_hat),
                         ("a    ", a_true, a_hat)]:
        r, p_r = pearsonr(vt, vh)
        rho, _ = spearmanr(vt, vh)
        print(f"{nome}  Pearson r = {r:+.4f}  (p={p_r:.2e})   Spearman = {rho:+.4f}")
        linhas.append((nome.strip(), r))

    r_log, _ = pearsonr(np.log(a_true), np.log(a_hat))
    print(f"a     Pearson r em log(a) = {r_log:+.4f}")

    print()
    print(f"theta:  true media={theta_true.mean():+.3f} sd={theta_true.std():.3f} | "
          f"est media={theta_hat.mean():+.3f} sd={theta_hat.std():.3f}")
    print(f"b:      true media={b_true.mean():+.3f} sd={b_true.std():.3f} | "
          f"est media={b_hat.mean():+.3f} sd={b_hat.std():.3f}")
    print(f"a:      true media={a_true.mean():.3f} sd={a_true.std():.3f} | "
          f"est media={a_hat.mean():.3f} sd={a_hat.std():.3f}")

    n_lim = int(np.sum((a_hat <= 0.06) | (a_hat >= 4.9) | (np.abs(b_hat) >= 5.9)))
    print(f"\nitens nos limites das bounds: {n_lim} / {len(a_hat)}")
    return dict(linhas)


res_sondagem = avaliar_recuperacao(theta_true, b_true, a_true,
                                   theta_hat, b_hat, a_hat,
                                   rotulo="sondagem 50 votos/item")

PASS_SONDAGEM = 0.80
veredicto = all(r >= PASS_SONDAGEM for r in res_sondagem.values())
print(f"\nveredicto sondagem (r >= {PASS_SONDAGEM} nos tres): "
      f"{'IMPLEMENTACAO CORRETA' if veredicto else 'FALHA — investigar'}")

--- recuperacao sondagem 50 votos/item ---
theta  Pearson r = +0.9722  (p=8.19e-127)   Spearman = +0.9863
b      Pearson r = +0.7984  (p=3.82e-222)   Spearman = +0.9019
a      Pearson r = +0.5509  (p=1.89e-80)   Spearman = +0.5616
a     Pearson r em log(a) = +0.5487

theta:  true media=-0.030 sd=0.880 | est media=+0.218 sd=1.083
b:      true media=-0.021 sd=1.005 | est media=+0.227 sd=1.887
a:      true media=1.021 sd=0.316 | est media=0.945 sd=0.617

itens nos limites das bounds: 50 / 1000

veredicto sondagem (r >= 0.8 nos tres): FALHA — investigar


In [ ]:
def diagnostico_escala(theta_hat, b_hat, a_hat, b_true, a_true, theta_true):
    lim = (a_hat <= 0.06) | (a_hat >= 4.9) | (np.abs(b_hat) >= 5.9)
    print(f"itens no bordo: {lim.sum()}")
    print(f"  destes, a no bordo inferior (<=0.06): {int(np.sum(a_hat <= 0.06))}")
    print(f"  a no bordo superior (>=4.9):          {int(np.sum(a_hat >= 4.9))}")
    print(f"  |b| no bordo (>=5.9):                 {int(np.sum(np.abs(b_hat) >= 5.9))}")

    ok = ~lim
    r_b, _ = pearsonr(b_true[ok], b_hat[ok])
    r_a, _ = pearsonr(a_true[ok], a_hat[ok])
    print(f"\nexcluindo itens no bordo (n={ok.sum()}):")
    print(f"  b  Pearson r = {r_b:+.4f}")
    print(f"  a  Pearson r = {r_a:+.4f}")

    s = theta_true.std() / theta_hat.std()
    d = theta_true.mean() - s * theta_hat.mean()
    th_t = s * theta_hat + d
    b_t  = s * b_hat + d
    a_t  = a_hat / s
    print(f"\napos alinhamento linear (escala={s:.4f}, desloc={d:+.4f}):")
    for nome, vt, vh in [("theta", theta_true, th_t), ("b", b_true, b_t), ("a", a_true, a_t)]:
        r, _ = pearsonr(vt, vh)
        print(f"  {nome:5s} r = {r:+.4f}   est media={vh.mean():+.3f} sd={vh.std():.3f}")

diagnostico_escala(theta_hat, b_hat, a_hat, b_true, a_true, theta_true)